In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

from jarvis.db.figshare import data as jarvis_data
from jarvis.core.atoms import Atoms

import sys

ROOT = Path(r"F:\Quantum-materials")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "data" / "enhanced_material_descriptors.csv"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"

RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("DATA:", DATA)

ROOT: F:\Quantum-materials
DATA: F:\Quantum-materials\data\enhanced_material_descriptors.csv


In [2]:
df = pd.read_csv(DATA)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (93902, 26)


,num_elements,total_atoms,mean_atomic_number,min_atomic_number,max_atomic_number,mean_atomic_mass,min_atomic_mass,max_atomic_mass,mean_atomic_radius,min_atomic_radius,...,electronegativity_difference,mean_ionization_energy,mean_electron_affinity,mean_s_valence,mean_p_valence,mean_d_valence,mean_f_valence,mean_period,mean_group,target_bandgap
0,4,4.0,24.500000,14.0,33.0,53.605025,28.085500,74.9216,1.250000,1.10,...,0.64,8.123695,0.844000,1.75,1.250000,5.500000,0.000000,3.750000,11.000000,0.000
1,2,7.0,13.714286,5.0,66.0,32.480857,10.811000,162.5000,0.978571,0.85,...,0.82,7.961003,0.300286,2.00,0.857143,0.000000,1.428571,2.857143,12.857143,0.000
2,3,4.0,32.000000,4.0,76.0,77.331091,9.012182,190.2300,1.175000,1.05,...,0.63,8.611033,0.518500,1.75,0.000000,3.250000,3.500000,3.750000,5.000000,0.000
3,2,2.0,51.000000,19.0,83.0,124.039350,39.098300,208.9804,1.900000,1.60,...,1.20,5.813082,0.698000,1.50,1.500000,5.000000,7.000000,5.000000,8.000000,0.472
4,2,3.0,30.333333,23.0,34.0,69.620500,50.941500,78.9600,1.216667,1.15,...,0.92,8.750323,1.468667,2.00,2.666667,7.666667,0.000000,4.000000,12.333333,0.000


In [3]:
print("Loading JARVIS-DFT 3D dataset...")

dft_3d = jarvis_data("dft_3d")

print("Number of JARVIS materials:", len(dft_3d))

Loading JARVIS-DFT 3D dataset...
Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.
Number of JARVIS materials: 93902


In [4]:
sample = dft_3d[0]

print(sample.keys())

dict_keys(['jid', 'spg_number', 'spg_symbol', 'formula', 'formation_energy_peratom', 'func', 'optb88vdw_bandgap', 'atoms', 'slme', 'magmom_oszicar', 'spillage', 'elastic_tensor', 'effective_masses_300K', 'kpoint_length_unit', 'maxdiff_mesh', 'maxdiff_bz', 'encut', 'optb88vdw_total_energy', 'epsx', 'epsy', 'epsz', 'mepsx', 'mepsy', 'mepsz', 'modes', 'magmom_outcar', 'max_efg', 'avg_elec_mass', 'avg_hole_mass', 'icsd', 'dfpt_piezo_max_eij', 'dfpt_piezo_max_dij', 'dfpt_piezo_max_dielectric', 'dfpt_piezo_max_dielectric_electronic', 'dfpt_piezo_max_dielectric_ionic', 'max_ir_mode', 'min_ir_mode', 'n-Seebeck', 'p-Seebeck', 'n-powerfact', 'p-powerfact', 'ncond', 'pcond', 'nkappa', 'pkappa', 'ehull', 'Tc_supercon', 'dimensionality', 'efg', 'xml_data_link', 'typ', 'exfoliation_energy', 'spg', 'crys', 'density', 'poisson', 'raw_files', 'nat', 'bulk_modulus_kv', 'shear_modulus_gv', 'mbj_bandgap', 'hse_gap', 'reference', 'search'])


In [5]:
print("JID:", sample.get("jid"))
print("Formula:", sample.get("formula"))
print("Atoms type:", type(sample.get("atoms")))

JID: JVASP-90856
Formula: TiCuSiAs
Atoms type: <class 'dict'>


In [6]:
atoms_data = sample.get("atoms")

print(atoms_data)

{'lattice_mat': [[3.566933224304235, 0.0, -0.0], [0.0, 3.566933224304235, -0.0], [-0.0, -0.0, 9.397075454186664]], 'coords': [[2.6751975000000003, 2.6751975000000003, 7.376101754328542], [0.8917325, 0.8917325, 2.0209782456714573], [0.8917325, 2.6751975000000003, 4.69854], [2.6751975000000003, 0.8917325, 4.69854], [0.8917325, 2.6751975000000003, 0.0], [2.6751975000000003, 0.8917325, 0.0], [2.6751975000000003, 2.6751975000000003, 2.8894795605846353], [0.8917325, 0.8917325, 6.507600439415366]], 'elements': ['Ti', 'Ti', 'Cu', 'Cu', 'Si', 'Si', 'As', 'As'], 'abc': [3.56693, 3.56693, 9.39708], 'angles': [90.0, 90.0, 90.0], 'cartesian': True, 'props': ['', '', '', '', '', '', '', '']}


In [7]:
jarvis_lookup = {
    item["jid"]: item
    for item in dft_3d
    if "jid" in item
}

print("JARVIS lookup size:", len(jarvis_lookup))

print("Example JIDs:")
print(list(jarvis_lookup.keys())[:5])

JARVIS lookup size: 93902
Example JIDs:
['JVASP-90856', 'JVASP-86097', 'JVASP-64906', 'JVASP-98225', 'JVASP-10']


In [9]:
# ---------------------------------------------------------
# Match enhanced dataset with JARVIS structures
# ---------------------------------------------------------

# JARVIS-DFT contains the material IDs.
# The enhanced CSV does not store JID, so create it from
# the JARVIS dataset instead of expecting df["jid"].

jarvis_lookup = {
    item["jid"]: item
    for item in dft_3d
    if "jid" in item
}

print("JARVIS lookup size:", len(jarvis_lookup))

# Check which column in the enhanced dataset identifies
# the material.
print("\nEnhanced dataset columns:")
print(df.columns.tolist())

JARVIS lookup size: 93902

Enhanced dataset columns:
['num_elements', 'total_atoms', 'mean_atomic_number', 'min_atomic_number', 'max_atomic_number', 'mean_atomic_mass', 'min_atomic_mass', 'max_atomic_mass', 'mean_atomic_radius', 'min_atomic_radius', 'max_atomic_radius', 'crys', 'spg_number', 'mean_electronegativity', 'min_electronegativity', 'max_electronegativity', 'electronegativity_difference', 'mean_ionization_energy', 'mean_electron_affinity', 'mean_s_valence', 'mean_p_valence', 'mean_d_valence', 'mean_f_valence', 'mean_period', 'mean_group', 'target_bandgap']


In [10]:
# ---------------------------------------------------------
# Find the material identifier in the enhanced dataset
# ---------------------------------------------------------

possible_id_columns = [
    col for col in df.columns
    if any(key in col.lower() for key in ["jid", "id", "material"])
]

print("Possible identifier columns:")
print(possible_id_columns)

Possible identifier columns:
[]


In [11]:
# ---------------------------------------------------------
# Verify row alignment between enhanced CSV and JARVIS
# ---------------------------------------------------------

print("Enhanced CSV rows :", len(df))
print("JARVIS rows      :", len(dft_3d))

if len(df) != len(dft_3d):
    raise ValueError(
        "Row counts do not match. We cannot safely attach "
        "JARVIS structures by row order."
    )

print("\nRow counts match: 93,902")
print("Now checking target alignment...")

# Compare the target band gap in the enhanced CSV
# with the JARVIS optb88vdw_bandgap values.

jarvis_bandgaps = np.array([
    item.get("optb88vdw_bandgap", np.nan)
    for item in dft_3d
], dtype=float)

csv_bandgaps = df["target_bandgap"].to_numpy(dtype=float)

valid = np.isfinite(jarvis_bandgaps) & np.isfinite(csv_bandgaps)

print("Comparable rows:", valid.sum())

if valid.sum() > 0:
    max_difference = np.max(
        np.abs(
            jarvis_bandgaps[valid] -
            csv_bandgaps[valid]
        )
    )

    mean_difference = np.mean(
        np.abs(
            jarvis_bandgaps[valid] -
            csv_bandgaps[valid]
        )
    )

    print(f"Maximum band-gap difference : {max_difference:.10f} eV")
    print(f"Mean band-gap difference    : {mean_difference:.10f} eV")

    if max_difference < 1e-6:
        print("\n✓ Row alignment confirmed.")
        print("✓ JARVIS structures can be safely attached by row order.")
    else:
        print("\n⚠ Target values do not perfectly match.")
        print("Do NOT proceed with row-wise structural descriptors.")

Enhanced CSV rows : 93902
JARVIS rows      : 93902

Row counts match: 93,902
Now checking target alignment...
Comparable rows: 93902
Maximum band-gap difference : 0.0000000000 eV
Mean band-gap difference    : 0.0000000000 eV

✓ Row alignment confirmed.
✓ JARVIS structures can be safely attached by row order.


In [12]:
# ---------------------------------------------------------
# Extract structure-aware descriptors from JARVIS
# ---------------------------------------------------------

structural_data = []

for i, item in enumerate(dft_3d):
    atoms = item.get("atoms")

    if atoms is None:
        structural_data.append({
            "num_sites": np.nan,
            "volume": np.nan,
            "volume_per_atom": np.nan,
            "density": np.nan,
            "lattice_a": np.nan,
            "lattice_b": np.nan,
            "lattice_c": np.nan,
            "lattice_alpha": np.nan,
            "lattice_beta": np.nan,
            "lattice_gamma": np.nan,
        })
        continue

    try:
        # Convert JARVIS atoms dictionary into an Atoms object
        atoms_obj = Atoms.from_dict(atoms)

        # Number of atomic sites
        num_sites = len(atoms_obj.elements)

        # Lattice information
        lattice = atoms_obj.lattice

        a = lattice.a
        b = lattice.b
        c = lattice.c

        alpha = lattice.alpha
        beta = lattice.beta
        gamma = lattice.gamma

        volume = lattice.volume

        # Volume per atomic site
        volume_per_atom = (
            volume / num_sites
            if num_sites > 0
            else np.nan
        )

        # Density
        density = atoms_obj.density

        structural_data.append({
            "num_sites": num_sites,
            "volume": volume,
            "volume_per_atom": volume_per_atom,
            "density": density,
            "lattice_a": a,
            "lattice_b": b,
            "lattice_c": c,
            "lattice_alpha": alpha,
            "lattice_beta": beta,
            "lattice_gamma": gamma,
        })

    except Exception as e:
        structural_data.append({
            "num_sites": np.nan,
            "volume": np.nan,
            "volume_per_atom": np.nan,
            "density": np.nan,
            "lattice_a": np.nan,
            "lattice_b": np.nan,
            "lattice_c": np.nan,
            "lattice_alpha": np.nan,
            "lattice_beta": np.nan,
            "lattice_gamma": np.nan,
        })

structural_df = pd.DataFrame(structural_data)

print("Structural descriptor shape:", structural_df.shape)
display(structural_df.head())

Structural descriptor shape: (93902, 10)


,num_sites,volume,volume_per_atom,density,lattice_a,lattice_b,lattice_c,lattice_alpha,lattice_beta,lattice_gamma
0,8,119.558951,14.944869,5.956102,3.566930,3.566930,9.397080,90.0000,90.0000,90.0000
1,7,68.371770,9.767396,5.522025,4.089080,4.089080,4.089080,90.0000,90.0000,90.0000
2,4,46.866605,11.716651,10.959730,4.343860,4.343860,4.343860,130.0642,130.0642,73.3042
3,32,1281.017367,40.031793,5.145217,7.296350,13.439606,14.224693,113.3104,90.0000,90.0000
4,3,60.658591,20.219530,5.717641,3.355502,3.355502,6.220810,90.0000,90.0000,120.0000


In [13]:
# ---------------------------------------------------------
# Validate structural descriptors
# ---------------------------------------------------------

print("Structural descriptor columns:")
print(structural_df.columns.tolist())

print("\nMissing values:")
print(structural_df.isna().sum())

print("\nSummary statistics:")
display(structural_df.describe().T)

Structural descriptor columns:
['num_sites', 'volume', 'volume_per_atom', 'density', 'lattice_a', 'lattice_b', 'lattice_c', 'lattice_alpha', 'lattice_beta', 'lattice_gamma']

Missing values:
num_sites          0
volume             0
volume_per_atom    0
density            0
lattice_a          0
lattice_b          0
lattice_c          0
lattice_alpha      0
lattice_beta       0
lattice_gamma      0
dtype: int64

Summary statistics:


,count,mean,std,min,25%,50%,75%,max
num_sites,93902.0,10.380301,8.995817,1.000000,4.000000,6.000000,14.000000,140.000000
volume,93902.0,182.627694,180.214753,5.655069,74.842843,131.823657,238.873982,8904.044000
volume_per_atom,93902.0,20.867425,77.563811,3.761959,12.994683,17.189094,23.771041,8000.000000
density,93902.0,6.384643,3.156905,0.000209,4.105746,5.836351,8.151145,24.266265
lattice_a,93902.0,5.264449,1.838361,0.993670,3.994140,4.919364,6.119754,34.594490
lattice_b,93902.0,5.564749,1.932214,1.389045,4.177085,5.261234,6.641660,30.169810
lattice_c,93902.0,7.042865,3.754613,0.994310,4.864473,6.252608,8.085375,82.519090
lattice_alpha,93902.0,86.431543,20.804178,9.127400,75.528025,90.000000,90.000000,166.671300
lattice_beta,93902.0,86.564564,20.561334,9.127400,76.854150,90.000000,90.000000,172.463700
lattice_gamma,93902.0,87.302589,24.207166,9.127400,61.599375,90.000000,101.758725,168.710000


In [14]:
# ---------------------------------------------------------
# Merge structural descriptors with the existing dataset
# ---------------------------------------------------------

df_structural = pd.concat(
    [
        df.reset_index(drop=True),
        structural_df.reset_index(drop=True)
    ],
    axis=1
)

print("Combined dataset shape:", df_structural.shape)

print("\nNumber of duplicate columns:")
print(
    df_structural.columns[
        df_structural.columns.duplicated()
    ].tolist()
)

print("\nCombined columns:")
print(df_structural.columns.tolist())

Combined dataset shape: (93902, 36)

Number of duplicate columns:
[]

Combined columns:
['num_elements', 'total_atoms', 'mean_atomic_number', 'min_atomic_number', 'max_atomic_number', 'mean_atomic_mass', 'min_atomic_mass', 'max_atomic_mass', 'mean_atomic_radius', 'min_atomic_radius', 'max_atomic_radius', 'crys', 'spg_number', 'mean_electronegativity', 'min_electronegativity', 'max_electronegativity', 'electronegativity_difference', 'mean_ionization_energy', 'mean_electron_affinity', 'mean_s_valence', 'mean_p_valence', 'mean_d_valence', 'mean_f_valence', 'mean_period', 'mean_group', 'target_bandgap', 'num_sites', 'volume', 'volume_per_atom', 'density', 'lattice_a', 'lattice_b', 'lattice_c', 'lattice_alpha', 'lattice_beta', 'lattice_gamma']


In [15]:
# ---------------------------------------------------------
# Inspect extreme structural values
# ---------------------------------------------------------

structural_features = [
    "num_sites",
    "volume",
    "volume_per_atom",
    "density",
    "lattice_a",
    "lattice_b",
    "lattice_c",
    "lattice_alpha",
    "lattice_beta",
    "lattice_gamma",
]

for feature in structural_features:
    s = df_structural[feature]

    print(
        f"{feature:20s} "
        f"min={s.min():10.3f} "
        f"median={s.median():10.3f} "
        f"max={s.max():10.3f}"
    )

num_sites            min=     1.000 median=     6.000 max=   140.000
volume               min=     5.655 median=   131.824 max=  8904.044
volume_per_atom      min=     3.762 median=    17.189 max=  8000.000
density              min=     0.000 median=     5.836 max=    24.266
lattice_a            min=     0.994 median=     4.919 max=    34.594
lattice_b            min=     1.389 median=     5.261 max=    30.170
lattice_c            min=     0.994 median=     6.253 max=    82.519
lattice_alpha        min=     9.127 median=    90.000 max=   166.671
lattice_beta         min=     9.127 median=    90.000 max=   172.464
lattice_gamma        min=     9.127 median=    90.000 max=   168.710


In [16]:
# ---------------------------------------------------------
# Inspect extreme volume-per-atom cases
# ---------------------------------------------------------

extreme = (
    df_structural
    .nlargest(20, "volume_per_atom")
    [
        [
            "volume",
            "num_sites",
            "volume_per_atom",
            "density",
            "target_bandgap"
        ]
    ]
)

display(extreme)

,volume,num_sites,volume_per_atom,density,target_bandgap
15138,8000.000000,1,8000.000000,0.005600,0.000
15211,8000.000000,1,8000.000000,0.007359,0.000
15229,8000.000000,1,8000.000000,0.000209,5.137
15259,8000.000000,1,8000.000000,0.004772,0.629
15309,8000.000000,1,8000.000000,0.006429,2.206
15367,8000.000000,1,8000.000000,0.006656,0.000
15478,8000.000000,1,8000.000000,0.005830,0.000
15514,8904.044000,2,4452.022000,0.010475,0.286
15335,8772.916000,2,4386.458000,0.011725,3.364
15398,8768.092000,2,4384.046000,0.012145,1.207


In [17]:
# ---------------------------------------------------------
# Define baseline and structure-aware feature sets
# ---------------------------------------------------------

baseline_features = [
    "num_elements",
    "total_atoms",
    "mean_atomic_number",
    "min_atomic_number",
    "max_atomic_number",
    "mean_atomic_mass",
    "min_atomic_mass",
    "max_atomic_mass",
    "mean_atomic_radius",
    "min_atomic_radius",
    "max_atomic_radius",
    "mean_electronegativity",
    "min_electronegativity",
    "max_electronegativity",
    "electronegativity_difference",
    "mean_ionization_energy",
    "mean_electron_affinity",
    "mean_s_valence",
    "mean_p_valence",
    "mean_d_valence",
    "mean_f_valence",
    "mean_period",
    "mean_group",
    "crys",
    "spg_number",
]

# The 10 newly extracted structure-aware descriptors
structural_features = [
    "num_sites",
    "volume",
    "volume_per_atom",
    "density",
    "lattice_a",
    "lattice_b",
    "lattice_c",
    "lattice_alpha",
    "lattice_beta",
    "lattice_gamma",
]

target = "target_bandgap"

print("Baseline features :", len(baseline_features))
print("Structural features:", len(structural_features))
print("Combined features :", len(baseline_features) + len(structural_features))

print("\nBaseline feature check:")
print(
    [f for f in baseline_features
     if f not in df_structural.columns]
)

print("\nStructural feature check:")
print(
    [f for f in structural_features
     if f not in df_structural.columns]
)

Baseline features : 25
Structural features: 10
Combined features : 35

Baseline feature check:
[]

Structural feature check:
[]


In [18]:
# ---------------------------------------------------------
# Verify the project's official feature list
# ---------------------------------------------------------

from src.preprocessing import FEATURE_COLUMNS

print("Official project feature count:", len(FEATURE_COLUMNS))
print("\nOfficial feature list:")
print(FEATURE_COLUMNS)

print("\nFeatures in our baseline but not official list:")
print(
    [f for f in baseline_features
     if f not in FEATURE_COLUMNS]
)

print("\nOfficial features missing from our baseline:")
print(
    [f for f in FEATURE_COLUMNS
     if f not in baseline_features]
)

Official project feature count: 25

Official feature list:
['num_elements', 'total_atoms', 'mean_atomic_number', 'min_atomic_number', 'max_atomic_number', 'mean_atomic_mass', 'min_atomic_mass', 'max_atomic_mass', 'mean_atomic_radius', 'min_atomic_radius', 'max_atomic_radius', 'mean_electronegativity', 'min_electronegativity', 'max_electronegativity', 'electronegativity_difference', 'mean_ionization_energy', 'mean_electron_affinity', 'mean_s_valence', 'mean_p_valence', 'mean_d_valence', 'mean_f_valence', 'mean_period', 'mean_group', 'crys', 'spg_number']

Features in our baseline but not official list:
[]

Official features missing from our baseline:
[]


In [19]:
# ---------------------------------------------------------
# Prepare baseline and structure-aware datasets
# ---------------------------------------------------------

X_baseline = df_structural[baseline_features].copy()
X_structural = df_structural[
    baseline_features + structural_features
].copy()

y = df_structural[target].copy()

print("Baseline X shape   :", X_baseline.shape)
print("Structural X shape :", X_structural.shape)
print("Target shape       :", y.shape)

print("\nTarget statistics:")
print(y.describe())

Baseline X shape   : (93902, 25)
Structural X shape : (93902, 35)
Target shape       : (93902,)

Target statistics:
count    93902.000000
mean         0.578425
std          1.310606
min          0.000000
25%          0.000000
50%          0.000000
75%          0.243000
max         18.179000
Name: target_bandgap, dtype: float64


In [20]:
# ---------------------------------------------------------
# Stratified train/test split
# ---------------------------------------------------------

y_class = (y > 0).astype(int)

Xb_train, Xb_test, y_train, y_test = train_test_split(
    X_baseline,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

Xs_train, Xs_test, _, _ = train_test_split(
    X_structural,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

print("Training samples:", len(y_train))
print("Test samples    :", len(y_test))

print("\nBaseline:")
print("Train:", Xb_train.shape)
print("Test :", Xb_test.shape)

print("\nStructure-aware:")
print("Train:", Xs_train.shape)
print("Test :", Xs_test.shape)

Training samples: 75121
Test samples    : 18781

Baseline:
Train: (75121, 25)
Test : (18781, 25)

Structure-aware:
Train: (75121, 35)
Test : (18781, 35)


In [21]:
# ---------------------------------------------------------
# Train baseline vs structure-aware Random Forest
# ---------------------------------------------------------

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

categorical_features = ["crys", "spg_number"]

baseline_numeric = [
    f for f in baseline_features
    if f not in categorical_features
]

structural_numeric = [
    f for f in baseline_features + structural_features
    if f not in categorical_features
]

def make_rf_pipeline(numeric_features):
    
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])


baseline_model = make_rf_pipeline(baseline_numeric)
structural_model = make_rf_pipeline(structural_numeric)

print("Training baseline model...")
baseline_model.fit(Xb_train, y_train)

print("Training structure-aware model...")
structural_model.fit(Xs_train, y_train)

print("Training complete.")

Training baseline model...
Training structure-aware model...
Training complete.


In [22]:
# ---------------------------------------------------------
# Compare baseline vs structure-aware model
# ---------------------------------------------------------

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

baseline_pred = baseline_model.predict(Xb_test)
structural_pred = structural_model.predict(Xs_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_pred)
)
baseline_r2 = r2_score(y_test, baseline_pred)

structural_mae = mean_absolute_error(y_test, structural_pred)
structural_rmse = np.sqrt(
    mean_squared_error(y_test, structural_pred)
)
structural_r2 = r2_score(y_test, structural_pred)

comparison = pd.DataFrame({
    "Model": [
        "Baseline (25 descriptors)",
        "Structure-aware (35 descriptors)"
    ],
    "MAE_eV": [
        baseline_mae,
        structural_mae
    ],
    "RMSE_eV": [
        baseline_rmse,
        structural_rmse
    ],
    "R2": [
        baseline_r2,
        structural_r2
    ]
})

display(comparison)

,Model,MAE_eV,RMSE_eV,R2
0,Baseline (25 descriptors),0.238135,0.555889,0.819530
1,Structure-aware (35 descriptors),0.246736,0.549728,0.823508


In [23]:
# ---------------------------------------------------------
# Calculate improvement from structural descriptors
# ---------------------------------------------------------

mae_improvement = (
    (baseline_mae - structural_mae)
    / baseline_mae
    * 100
)

rmse_improvement = (
    (baseline_rmse - structural_rmse)
    / baseline_rmse
    * 100
)

r2_change = structural_r2 - baseline_r2

print(f"MAE improvement  : {mae_improvement:.2f}%")
print(f"RMSE improvement : {rmse_improvement:.2f}%")
print(f"R² change        : {r2_change:+.4f}")

MAE improvement  : -3.61%
RMSE improvement : 1.11%
R² change        : +0.0040


In [24]:
# ---------------------------------------------------------
# Create chemical-system groups
# ---------------------------------------------------------

def get_chemical_system(formula):
    import re

    elements = re.findall(r"[A-Z][a-z]?", formula)
    return "-".join(sorted(set(elements)))


# Reconstruct the formulas from JARVIS.
# Row alignment was already verified above.
formulas = [
    item.get("formula", "")
    for item in dft_3d
]

groups = np.array([
    get_chemical_system(formula)
    for formula in formulas
])

print("Number of materials:", len(groups))
print("Unique chemical systems:", len(np.unique(groups)))

print("\nExample chemical systems:")
print(groups[:20])

Number of materials: 93902
Unique chemical systems: 32707

Example chemical systems:
['As-Cu-Si-Ti' 'B-Dy' 'Be-Os-Ru' 'Bi-K' 'Se-V' 'Mn-Si-Tb' 'Ba-Bi-Na'
 'Fe-O-Sr' 'Lu-Ni-Sn' 'Mo-S-Se-W' 'Ba-Ce-Mn-O' 'Fe-O-Si' 'Ba-Li-N-Na'
 'Ag-I-O' 'Cr-Li-Mn-O' 'As-Cd-O' 'F-Fe-O' 'Cu-Na-O' 'Os-Sb-Sm' 'Mo-S-Se-W']


In [25]:
# ---------------------------------------------------------
# Verify chemical-system groups
# ---------------------------------------------------------

print("Group array length:", len(groups))
print("Dataset length    :", len(df_structural))

assert len(groups) == len(df_structural)

print("✓ Chemical-system groups align with the dataset.")

Group array length: 93902
Dataset length    : 93902
✓ Chemical-system groups align with the dataset.


In [26]:
# ---------------------------------------------------------
# Chemical-system GroupKFold comparison
# ---------------------------------------------------------

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

gkf = GroupKFold(n_splits=5)

baseline_results = []
structural_results = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X_baseline, y, groups=groups),
    start=1
):

    print(f"\n========== Fold {fold} ==========")

    # -----------------------------
    # Baseline
    # -----------------------------

    X_train_b = X_baseline.iloc[train_idx]
    X_test_b = X_baseline.iloc[test_idx]

    y_train_fold = y.iloc[train_idx]
    y_test_fold = y.iloc[test_idx]

    baseline_fold_model = make_rf_pipeline(
        baseline_numeric
    )

    baseline_fold_model.fit(
        X_train_b,
        y_train_fold
    )

    pred_b = baseline_fold_model.predict(
        X_test_b
    )

    mae_b = mean_absolute_error(
        y_test_fold,
        pred_b
    )

    rmse_b = np.sqrt(
        mean_squared_error(
            y_test_fold,
            pred_b
        )
    )

    r2_b = r2_score(
        y_test_fold,
        pred_b
    )

    baseline_results.append(
        [fold, mae_b, rmse_b, r2_b]
    )

    # -----------------------------
    # Structure-aware
    # -----------------------------

    X_train_s = X_structural.iloc[train_idx]
    X_test_s = X_structural.iloc[test_idx]

    structural_fold_model = make_rf_pipeline(
        structural_numeric
    )

    structural_fold_model.fit(
        X_train_s,
        y_train_fold
    )

    pred_s = structural_fold_model.predict(
        X_test_s
    )

    mae_s = mean_absolute_error(
        y_test_fold,
        pred_s
    )

    rmse_s = np.sqrt(
        mean_squared_error(
            y_test_fold,
            pred_s
        )
    )

    r2_s = r2_score(
        y_test_fold,
        pred_s
    )

    structural_results.append(
        [fold, mae_s, rmse_s, r2_s]
    )

    print(
        f"Baseline   : "
        f"MAE={mae_b:.4f}, "
        f"RMSE={rmse_b:.4f}, "
        f"R²={r2_b:.4f}"
    )

    print(
        f"Structural : "
        f"MAE={mae_s:.4f}, "
        f"RMSE={rmse_s:.4f}, "
        f"R²={r2_s:.4f}"
    )


========== Fold 1 ==========
Baseline   : MAE=0.3033, RMSE=0.6566, R²=0.7563
Structural : MAE=0.2991, RMSE=0.6397, R²=0.7687

========== Fold 2 ==========
Baseline   : MAE=0.3243, RMSE=0.7248, R²=0.7140
Structural : MAE=0.3214, RMSE=0.7064, R²=0.7283

========== Fold 3 ==========
Baseline   : MAE=0.3243, RMSE=0.7218, R²=0.7190
Structural : MAE=0.3085, RMSE=0.6548, R²=0.7688

========== Fold 4 ==========
Baseline   : MAE=0.2967, RMSE=0.6416, R²=0.7433
Structural : MAE=0.2984, RMSE=0.6380, R²=0.7461

========== Fold 5 ==========
Baseline   : MAE=0.3061, RMSE=0.6415, R²=0.7296
Structural : MAE=0.2945, RMSE=0.6167, R²=0.7502


In [27]:
# ---------------------------------------------------------
# Summarize chemical-system GroupKFold results
# ---------------------------------------------------------

baseline_results_df = pd.DataFrame(
    baseline_results,
    columns=["fold", "MAE", "RMSE", "R2"]
)

structural_results_df = pd.DataFrame(
    structural_results,
    columns=["fold", "MAE", "RMSE", "R2"]
)

print("BASELINE — Chemical-system GroupKFold")
print(
    f"MAE  : {baseline_results_df['MAE'].mean():.4f} "
    f"+/- {baseline_results_df['MAE'].std():.4f}"
)
print(
    f"RMSE : {baseline_results_df['RMSE'].mean():.4f} "
    f"+/- {baseline_results_df['RMSE'].std():.4f}"
)
print(
    f"R²   : {baseline_results_df['R2'].mean():.4f} "
    f"+/- {baseline_results_df['R2'].std():.4f}"
)

print("\nSTRUCTURE-AWARE — Chemical-system GroupKFold")
print(
    f"MAE  : {structural_results_df['MAE'].mean():.4f} "
    f"+/- {structural_results_df['MAE'].std():.4f}"
)
print(
    f"RMSE : {structural_results_df['RMSE'].mean():.4f} "
    f"+/- {structural_results_df['RMSE'].std():.4f}"
)
print(
    f"R²   : {structural_results_df['R2'].mean():.4f} "
    f"+/- {structural_results_df['R2'].std():.4f}"
)

BASELINE — Chemical-system GroupKFold
MAE  : 0.3109 +/- 0.0127
RMSE : 0.6773 +/- 0.0425
R²   : 0.7324 +/- 0.0174

STRUCTURE-AWARE — Chemical-system GroupKFold
MAE  : 0.3044 +/- 0.0108
RMSE : 0.6511 +/- 0.0338
R²   : 0.7524 +/- 0.0170


In [28]:
# ---------------------------------------------------------
# Structural descriptor improvement
# under chemical-system GroupKFold
# ---------------------------------------------------------

baseline_mae_gkf = baseline_results_df["MAE"].mean()
structural_mae_gkf = structural_results_df["MAE"].mean()

baseline_rmse_gkf = baseline_results_df["RMSE"].mean()
structural_rmse_gkf = structural_results_df["RMSE"].mean()

baseline_r2_gkf = baseline_results_df["R2"].mean()
structural_r2_gkf = structural_results_df["R2"].mean()

mae_change = (
    (structural_mae_gkf - baseline_mae_gkf)
    / baseline_mae_gkf * 100
)

rmse_change = (
    (structural_rmse_gkf - baseline_rmse_gkf)
    / baseline_rmse_gkf * 100
)

r2_change = structural_r2_gkf - baseline_r2_gkf

print(f"MAE change  : {mae_change:+.2f}%")
print(f"RMSE change : {rmse_change:+.2f}%")
print(f"R² change   : {r2_change:+.4f}")

MAE change  : -2.10%
RMSE change : -3.86%
R² change   : +0.0200


In [29]:
# ---------------------------------------------------------
# Structural feature importance
# ---------------------------------------------------------

from sklearn.inspection import permutation_importance

# Use the final random-split structure-aware model
# for an initial diagnostic.
#
# This tells us which structural descriptors affect
# predictions on unseen test materials.

perm = permutation_importance(
    structural_model,
    Xs_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": Xs_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
})

importance_df = (
    importance_df
    .sort_values(
        "importance_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top structural-model features:")
display(importance_df.head(20))

Top structural-model features:


,feature,importance_mean,importance_std
0,mean_ionization_energy,0.303245,0.002379
1,mean_d_valence,0.212056,0.003070
2,mean_p_valence,0.138591,0.001669
3,max_electronegativity,0.115659,0.001806
4,volume,0.099365,0.001246
5,num_sites,0.098703,0.002123
6,density,0.072617,0.000402
7,mean_group,0.071810,0.001183
8,mean_electron_affinity,0.067688,0.000541
9,mean_electronegativity,0.058215,0.000805


In [30]:
# ---------------------------------------------------------
# Structural descriptors only
# ---------------------------------------------------------

structural_importance = (
    importance_df[
        importance_df["feature"].isin(
            structural_features
        )
    ]
    .copy()
)

display(structural_importance)

,feature,importance_mean,importance_std
4,volume,0.099365,0.001246
5,num_sites,0.098703,0.002123
6,density,0.072617,0.000402
10,volume_per_atom,0.056981,0.000893
23,lattice_b,0.016782,0.000447
25,lattice_c,0.013083,0.000307
26,lattice_a,0.012540,0.000354
27,lattice_alpha,0.012464,0.000598
29,lattice_beta,0.009335,0.000327
34,lattice_gamma,0.005389,0.000202


In [31]:
# ---------------------------------------------------------
# Save structural feature importance
# ---------------------------------------------------------

output_file = (
    RESULTS /
    "structural_feature_importance.csv"
)

structural_importance.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: F:\Quantum-materials\results\structural_feature_importance.csv


In [35]:
import pandas as pd

print("=" * 70)
print("DATAFRAMES CURRENTLY IN NOTEBOOK 08")
print("=" * 70)

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        print(f"\n{name}")
        print("-" * 70)
        print("Shape:", obj.shape)

        print("Columns:")
        print(
            obj.columns.tolist()
        )

DATAFRAMES CURRENTLY IN NOTEBOOK 08

df
----------------------------------------------------------------------
Shape: (93902, 26)
Columns:
['num_elements', 'total_atoms', 'mean_atomic_number', 'min_atomic_number', 'max_atomic_number', 'mean_atomic_mass', 'min_atomic_mass', 'max_atomic_mass', 'mean_atomic_radius', 'min_atomic_radius', 'max_atomic_radius', 'crys', 'spg_number', 'mean_electronegativity', 'min_electronegativity', 'max_electronegativity', 'electronegativity_difference', 'mean_ionization_energy', 'mean_electron_affinity', 'mean_s_valence', 'mean_p_valence', 'mean_d_valence', 'mean_f_valence', 'mean_period', 'mean_group', 'target_bandgap']

structural_df
----------------------------------------------------------------------
Shape: (93902, 10)
Columns:
['num_sites', 'volume', 'volume_per_atom', 'density', 'lattice_a', 'lattice_b', 'lattice_c', 'lattice_alpha', 'lattice_beta', 'lattice_gamma']

df_structural
----------------------------------------------------------------------

In [36]:
# ============================================================
# SAVE FINAL NOTEBOOK 08 DATASET
# ============================================================

from pathlib import Path

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    DATA_DIR /
    "structural_enhanced_features.csv"
)

df_structural.to_csv(
    output_file,
    index=False
)

print("Dataset saved successfully!")
print()
print("File:")
print(output_file)
print()
print("Shape:")
print(df_structural.shape)
print()
print("Columns:")
print(df_structural.columns.tolist())
print()
print(
    "Missing values:",
    df_structural.isna().sum().sum()
)

Dataset saved successfully!

File:
F:\Quantum-materials\data\structural_enhanced_features.csv

Shape:
(93902, 36)

Columns:
['num_elements', 'total_atoms', 'mean_atomic_number', 'min_atomic_number', 'max_atomic_number', 'mean_atomic_mass', 'min_atomic_mass', 'max_atomic_mass', 'mean_atomic_radius', 'min_atomic_radius', 'max_atomic_radius', 'crys', 'spg_number', 'mean_electronegativity', 'min_electronegativity', 'max_electronegativity', 'electronegativity_difference', 'mean_ionization_energy', 'mean_electron_affinity', 'mean_s_valence', 'mean_p_valence', 'mean_d_valence', 'mean_f_valence', 'mean_period', 'mean_group', 'target_bandgap', 'num_sites', 'volume', 'volume_per_atom', 'density', 'lattice_a', 'lattice_b', 'lattice_c', 'lattice_alpha', 'lattice_beta', 'lattice_gamma']

Missing values: 537
